In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

### Temperature Logger Calibration

In [3]:
B1_TL = pd.read_csv('../data/groundwater/Temp/Calibration/BA0000007FD53E41_101326145001_1.csv', skiprows=24, on_bad_lines='skip', sep=';')
B2_TL = pd.read_csv('../data/groundwater/Temp/Calibration/AE000000824F6A41_101326145001_1.csv', skiprows=24, on_bad_lines='skip', sep=';')
B3_TL = pd.read_csv('../data/groundwater/Temp/Calibration/23000000824F2141_101326145001_1.csv', skiprows=24, on_bad_lines='skip', sep=';')
A_TL = pd.read_csv('../data/groundwater/Temp/Calibration/9600000082456D41_101326145001_1.csv', skiprows=24, on_bad_lines='skip', sep=';')

start = pd.Timestamp('2026-10-13 08:50:01')
end = pd.Timestamp('2026-10-13 09:10:01')

stats_TL = pd.DataFrame()

for name, data in zip(['A_TL', 'B1_TL', 'B2_TL', 'B3_TL'], [A_TL, B1_TL, B2_TL, B3_TL]):
    data['Date'] = pd.to_datetime(data['Date'] + ' ' + data['Time'])
    data['Value'] = pd.to_numeric(data['Value'], errors='coerce')
    data = data.loc[data['Date'].between(start, end)].drop(columns='Time')
    stats_TL[name] = data['Value'].describe().loc[['mean', 'std', 'min', '50%', 'max']]

stats_TL.round(2)


,A_TL,B1_TL,B2_TL,B3_TL
mean,26.42,26.58,26.42,26.48
std,0.23,0.21,0.22,0.23
min,26.06,26.25,26.06,26.12
50%,26.44,26.56,26.44,26.50
max,26.69,26.88,26.69,26.75


### Pressure Logger Calibration

In [4]:
B1 = pd.read_csv('../data/groundwater/DiverCSV/Calibration/B1_261013121155_AL116.CSV', encoding="latin-1", skiprows=51, usecols=[0, 1], delimiter=';')[:-1]
B2 = pd.read_csv('../data/groundwater/DiverCSV/Calibration/B2_261013121326_DF001.CSV', encoding="latin-1", skiprows=51, usecols=[0, 1], delimiter=';')[:-1]
B3 = pd.read_csv('../data/groundwater/DiverCSV/Calibration/B3_261013121404_AL207.CSV', encoding="latin-1", skiprows=51, usecols=[0, 1], delimiter=';')[:-1]
A1 = pd.read_csv('../data/groundwater/DiverCSV/Calibration/A1_261013121508_1358A.CSV', encoding="latin-1", skiprows=51, usecols=[0, 1], delimiter=';')[:-1]
Baro = pd.read_csv('../data/groundwater/DiverCSV/Calibration/Baro B_261013121439_Y3119.CSV', encoding="latin-1", skiprows=51, usecols=[0, 1], delimiter=';')[:-1]

start = pd.Timestamp('2026-10-13 11:27:00.0')
end = pd.Timestamp('2026-10-13 11:42:00.0')

start_w = pd.Timestamp('2026-10-13 11:45:00.0')
end_w = pd.Timestamp('2026-10-13 12:00:00.0')

for name, data in zip(['B1', 'B2', 'B3', 'A1', 'Baro'], [B1, B2, B3, A1, Baro]):
    data['Date/time'] = pd.to_datetime(data.iloc[:, 0], format='mixed', dayfirst=True)
    data['Pressure[cmH2O]'] = pd.to_numeric(data['Pressure[cmH2O]'].astype(str).str.replace(r',(?=.*,)', '', regex=True).str.replace(',', '.'), errors='coerce')
    filtered = data.loc[data['Date/time'].between(start, end)].copy()
    filtered_w = data.loc[data['Date/time'].between(start_w, end_w)].copy()
    
    if name == 'B1':
        B1_w, B1 = filtered_w, filtered
    elif name == 'B2':
        B2_w, B2 = filtered_w, filtered
    elif name == 'B3':
        B3_w, B3 = filtered_w, filtered
    elif name == 'A1':
        A1_w, A1 = filtered_w, filtered
    elif name == 'Baro':
        Baro_w, Baro = filtered_w, filtered

In [ ]:
stats_a = pd.DataFrame()
stats_w = pd.DataFrame()

for name, norm, water in zip(['B1', 'B2', 'B3', 'A1', 'Baro'], [B1, B2, B3, A1, Baro], [B1_w, B2_w, B3_w, A1_w, Baro_w]):
    stats_a[name] = norm['Pressure[cmH2O]'].describe().loc[['mean', 'std', 'min', '50%', 'max']]
    stats_w[name] = water['Pressure[cmH2O]'].describe().loc[['mean', 'std', 'min', '50%', 'max']]

mean_a = stats_a.loc['mean']
mean_w = stats_w.loc['mean']

offsets = pd.DataFrame({'Mean Pressure in Air (cmH2O)':    mean_a,
                        'Offset in Air (cmH2O)':          mean_a - mean_a.mean(),
                        'Mean Pressure in Water (cmH2O)':  mean_w,
                        'Offset in Water (cmH2O)':        mean_w - mean_w.mean()})

offsets['Mean Offset'] = (offsets['Offset in Water (cmH2O)'] + offsets['Offset in Air (cmH2O)']) / 2

n_a = pd.Series({n: d['Pressure[cmH2O]'].count() for n, d in zip(['B1','B2','B3','A1','Baro'], [B1, B2, B3, A1, Baro])})
n_w = pd.Series({n: d['Pressure[cmH2O]'].count() for n, d in zip(['B1','B2','B3','A1','Baro'], [B1_w, B2_w, B3_w, A1_w, Baro_w])})
offsets['Air SE']   = stats_a.loc['std'] / np.sqrt(n_a)
offsets['Water SE'] = stats_w.loc['std'] / np.sqrt(n_w)

offsets['Water level [cm]'] = (offsets['Mean Pressure in Water (cmH2O)'] - offsets['Mean Pressure in Air (cmH2O)'])
offsets.round(3)



,Mean Pressure in Air (cmH2O),Offset in Air (cmH2O),Mean Pressure in Water (cmH2O),Offset in Water (cmH2O),Mean Offset,Air SE,Water SE,Water level [m]
B1,1032.074,0.806,1041.354,0.970,0.888,0.026,0.027,9.280
B2,1032.937,1.669,1041.919,1.534,1.602,0.023,0.034,8.981
B3,1031.358,0.090,1040.380,-0.005,0.043,0.025,0.032,9.021
A1,1029.624,-1.644,1038.714,-1.670,-1.657,0.023,0.020,9.090
Baro,1030.346,-0.922,1039.555,-0.829,-0.875,0.024,0.025,9.209
